# 02 · Model quality on four dimensions — Colab-safe version

Computes **fitness, precision, generalization and simplicity** for Alpha,
Heuristic and Inductive Miner under both case notions.

## Why this version is Colab-safe

The original all-faculty run can exceed a Colab session because ETC precision is
computationally expensive. This version therefore:

1. runs **one faculty at a time** (both roles by default);
2. checkpoints after **every miner** to Google Drive;
3. resumes automatically after a disconnect;
4. skips ETC precision on **user-level traces by default**, because the revision
   already documents that it is computationally prohibitive for these long traces;
5. keeps course-level precision enabled for the comparable four-dimensional evaluation.

Open this notebook separately for each faculty, change `FOCUS_FACULTY`, and click
**Runtime → Run all**. A new Colab runtime is fine because results are stored in
`MyDrive/ProcessMining/results/`.

## 1. Setup

Run this section first. It installs dependencies and downloads the deposit from
figshare into the Colab VM.

**Runtime:** Runtime &rarr; Change runtime type &rarr; **High-RAM** if available.
The largest faculty file (FIF, 1.3 GB on disk) needs roughly 6 GB once loaded.

In [1]:

!pip install -q pm4py==2.7.23.3 statsmodels psutil 2>/dev/null

import os, warnings, sys, json, time, gc, random, pathlib
os.environ["TQDM_DISABLE"] = "1"
warnings.filterwarnings("ignore")
import pandas as pd, numpy as np

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    RESULTS_DIR = "/content/drive/MyDrive/ProcessMining/results"
except Exception:
    RESULTS_DIR = "/content/ProcessMining/results"
pathlib.Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

print("Python ", sys.version.split()[0])
print("pandas ", pd.__version__)
import pm4py; print("pm4py  ", pm4py.__version__)
print("results", RESULTS_DIR)

try:
    import psutil
    gb = psutil.virtual_memory().total / 1e9
    print(f"RAM     {gb:.1f} GB")
    if gb < 20:
        print("NOTE: standard runtime detected. One-faculty batching is strongly recommended.")
except Exception:
    pass

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Python  3.13.15
pandas  2.2.3




  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




pm4py   2.7.23.3
results /content/drive/MyDrive/ProcessMining/results
RAM     13.6 GB
NOTE: standard runtime detected. One-faculty batching is strongly recommended.


In [2]:




import requests, os, pathlib

ARTICLE = "28341992"
DATA_DIR = "/content/data"
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

meta = requests.get(f"https://api.figshare.com/v2/articles/{ARTICLE}", timeout=60).json()
print(f"{meta['title']}  (v{meta.get('version','?')})")
print(f"{len(meta['files'])} files, {meta['size']/1e9:.2f} GB total\n")

FILES = {}
for f in meta["files"]:
    FILES[f["name"]] = f["download_url"]
    print(f"  {f['name']:<32} {f['size']/1e6:>8.1f} MB")


FACULTIES = ["FEB", "FIF", "FIK", "FIT", "FKB", "FRI", "FTE"]
missing = [f"{fac}_{role}.csv" for fac in FACULTIES
           for role in ("Student", "Lecturer") if f"{fac}_{role}.csv" not in FILES]
if missing:
    print("\n  MISSING FROM DEPOSIT:")
    for m in missing:
        print(f"    {m}")
    print("\n  Analyses for these partitions will be skipped.")


def fetch(name):
    """Download one file if not already present. Returns local path or None."""
    if name not in FILES:
        return None
    dest = os.path.join(DATA_DIR, name)
    if os.path.exists(dest) and os.path.getsize(dest) > 1_000_000:
        return dest
    print(f"downloading {name} ...", flush=True)
    with requests.get(FILES[name], stream=True, timeout=1800) as r:
        r.raise_for_status()
        with open(dest, "wb") as fh:
            for chunk in r.iter_content(1 << 22):
                fh.write(chunk)
    print(f"  -> {os.path.getsize(dest)/1e6:.0f} MB")
    return dest

Process Mining in CeLOE  (v2)
7 files, 5.21 GB total

  FIF_Student.csv                    1314.5 MB
  FIT_Student.csv                     303.8 MB
  FIK_Student.csv                     417.6 MB
  FEB_Student.csv                     881.4 MB
  FTE_Student.csv                     874.3 MB
  FRI_Student.csv                     753.6 MB
  FKB_Student.csv                     666.4 MB

  MISSING FROM DEPOSIT:
    FEB_Lecturer.csv
    FIF_Lecturer.csv
    FIK_Lecturer.csv
    FIT_Lecturer.csv
    FKB_Lecturer.csv
    FRI_Lecturer.csv
    FTE_Lecturer.csv

  Analyses for these partitions will be skipped.


In [3]:



USECOLS = ["id", "eventname", "component", "action", "target", "crud",
           "edulevel", "userid", "courseid", "timecreated", "event"]
DTYPES = {"id": "int64", "eventname": "category", "component": "category",
          "action": "category", "target": "category", "crud": "category",
          "edulevel": "int8", "userid": "int32", "courseid": "int32",
          "event": "category"}

CUTOFF = "2023-06-26"


def load(faculty, role, apply_dedup=True, cols=None):
    """Load one faculty-role partition.

    apply_dedup fixes the defect found during revision: the original notebooks
    called df.drop_duplicates() WITHOUT assignment, so duplicates were counted
    and reported but never removed from the working data.
    """
    name = f"{faculty}_{role}.csv"
    path = fetch(name)
    if path is None:
        print(f"  [skip] {name} not in deposit")
        return None
    use = cols or USECOLS
    df = pd.read_csv(path, index_col=0, usecols=lambda c: c in use or c == "Unnamed: 0",
                     dtype={k: v for k, v in DTYPES.items() if k in use},
                     parse_dates=["timecreated"] if "timecreated" in use else None)
    n_raw = len(df)
    n_dup = int(df.duplicated().sum())
    if apply_dedup and n_dup:
        df = df.drop_duplicates()
    df.attrs["n_raw"] = n_raw
    df.attrs["n_dup"] = n_dup
    df.attrs["faculty"] = faculty
    df.attrs["role"] = role
    return df


def add_case(df, notion):
    """notion is 'user' or 'course'."""
    if notion == "user":
        df["case"] = df["userid"].astype(str)
    else:
        df["case"] = df["userid"].astype(str) + "_" + df["courseid"].astype(str)
    return df


def to_log(df, notion, sample=None, seed=42, max_len=None):
    """Build a pm4py EventLog. sample caps the number of traces."""
    d = add_case(df, notion)
    if max_len:
        keep = d.groupby("case").size()
        d = d[d["case"].isin(keep[keep <= max_len].index)]
    if sample:
        random.seed(seed)
        cases = sorted(d["case"].unique())
        d = d[d["case"].isin(set(random.sample(cases, min(sample, len(cases)))))]
    ldf = (d[["case", "event", "timecreated"]]
           .rename(columns={"case": "case:concept:name", "event": "concept:name",
                            "timecreated": "time:timestamp"})
           .sort_values(["case:concept:name", "time:timestamp"])
           .reset_index(drop=True))

    ldf["case:concept:name"] = ldf["case:concept:name"].astype(str)
    ldf["concept:name"] = ldf["concept:name"].astype(str)
    return pm4py.convert_to_event_log(ldf), ldf



def save(obj, name):
    """Persist results in Google Drive (or local fallback outside Colab)."""
    if isinstance(obj, pd.DataFrame):
        path = os.path.join(RESULTS_DIR, f"{name}.csv")
        obj.to_csv(path, index=False)
    else:
        path = os.path.join(RESULTS_DIR, f"{name}.json")
        with open(path, "w") as fh:
            json.dump(obj, fh, indent=1)
    print(f"saved {path}")
    return path


## 2. Configuration

In [4]:

FOCUS_FACULTY = "FIF"
ROLE_MODE = "Both"


SAMPLE_TRACES_COURSE = 75
SAMPLE_TRACES_USER   = 40
MAX_TRACE_LEN        = 2000
SEED                 = 42


COMPUTE_USER_PRECISION = False

RUN_FACULTIES = [FOCUS_FACULTY]
RUN_ROLES = ["Student", "Lecturer"] if ROLE_MODE == "Both" else [ROLE_MODE]
RUN_NOTIONS = ["course", "user"]

from pm4py.algo.discovery.alpha import algorithm as alpha_miner
from pm4py.algo.discovery.heuristics import algorithm as heuristics_miner
from pm4py.algo.discovery.inductive import algorithm as inductive_miner
from pm4py.algo.evaluation.replay_fitness import algorithm as fit_eval
from pm4py.algo.evaluation.precision import algorithm as prec_eval
from pm4py.algo.evaluation.generalization import algorithm as gen_eval
from pm4py.algo.evaluation.simplicity import algorithm as sim_eval

P = {"show_progress_bar": False}

MINER_VARIANTS = {
    "alpha": "PM4Py Alpha default",
    "heuristic": "Variants.CLASSIC",
    "inductive": "Variants.IM",
}

def discover(log, miner):
    if miner == "alpha":
        return alpha_miner.apply(log)
    if miner == "heuristic":
        return heuristics_miner.apply(log, variant=heuristics_miner.Variants.CLASSIC)
    t = inductive_miner.apply(log, variant=inductive_miner.Variants.IM)
    return pm4py.convert_to_petri_net(t) if not isinstance(t, tuple) else t

metadata = {
    "python": sys.version,
    "pandas": pd.__version__,
    "pm4py": pm4py.__version__,
    "faculty_this_run": FOCUS_FACULTY,
    "roles_this_run": RUN_ROLES,
    "sample_traces_course": SAMPLE_TRACES_COURSE,
    "sample_traces_user": SAMPLE_TRACES_USER,
    "max_trace_len": MAX_TRACE_LEN,
    "seed": SEED,
    "compute_user_precision": COMPUTE_USER_PRECISION,
    "miner_variants": MINER_VARIANTS,
    "analysis_window_note": "Deposited role partitions are already restricted to dates before 2023-06-26."
}
save(metadata, f"02_run_metadata_{FOCUS_FACULTY}")
print(json.dumps(metadata, indent=2))

saved /content/drive/MyDrive/ProcessMining/results/02_run_metadata_FIF.json
{
  "python": "3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]",
  "pandas": "2.2.3",
  "pm4py": "2.7.23.3",
  "faculty_this_run": "FIF",
  "roles_this_run": [
    "Student",
    "Lecturer"
  ],
  "sample_traces_course": 75,
  "sample_traces_user": 40,
  "max_trace_len": 2000,
  "seed": 42,
  "compute_user_precision": false,
  "miner_variants": {
    "alpha": "PM4Py Alpha default",
    "heuristic": "Variants.CLASSIC",
    "inductive": "Variants.IM"
  },
  "analysis_window_note": "Deposited role partitions are already restricted to dates before 2023-06-26."
}


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Run

Writes each result to `/content/02_quality.csv` as it goes, so a disconnect does
not lose completed work. Re-running skips combinations already present.

In [6]:
OUT = os.path.join(RESULTS_DIR, f"02_quality_{FOCUS_FACULTY}.csv")
done = set()
if os.path.exists(OUT):
    prev = pd.read_csv(OUT)
    done = {(r.faculty, r.role, r.notion, r.miner) for r in prev.itertuples()}
    print(f"resuming {FOCUS_FACULTY}: {len(done)} combinations already computed from {OUT}")

results = []

def flush():
    if not results:
        return
    d = pd.DataFrame(results)
    if os.path.exists(OUT):
        d = pd.concat([pd.read_csv(OUT), d], ignore_index=True)
    d.drop_duplicates(subset=["faculty", "role", "notion", "miner"], keep="last").to_csv(OUT, index=False)

for fac in RUN_FACULTIES:
    for role in RUN_ROLES:
        df = None
        for notion in RUN_NOTIONS:
            todo = [m for m in ("alpha", "heuristic", "inductive")
                    if (fac, role, notion, m) not in done]
            if not todo:
                print(f"[already done] {fac} {role} {notion}")
                continue
            if df is None:
                df = load(fac, role)
                if df is None:
                    break
            n = SAMPLE_TRACES_COURSE if notion == "course" else SAMPLE_TRACES_USER
            log, ldf = to_log(df, notion, sample=n, seed=SEED, max_len=MAX_TRACE_LEN)
            tl = ldf.groupby("case:concept:name").size()
            print(f"\n[{fac} {role} {notion}] {len(log):,} traces, {len(ldf):,} events, mean len {tl.mean():.0f}", flush=True)

            for miner in todo:
                r = dict(faculty=fac, role=role, notion=notion, miner=miner,
                         traces=len(log), events=len(ldf),
                         activities=int(ldf["concept:name"].nunique()),
                         mean_len=round(float(tl.mean()), 1),
                         sample_requested=n, seed=SEED, max_trace_len=MAX_TRACE_LEN)
                try:
                    t0 = time.time(); net, im, fm = discover(log, miner)
                    r.update(disc_s=round(time.time() - t0, 1),
                             places=len(net.places), transitions=len(net.transitions),
                             arcs=len(net.arcs),
                             silent=sum(1 for x in net.transitions if x.label is None),
                             simplicity=round(sim_eval.apply(net), 4))
                    r["fitness"] = round(fit_eval.apply(log, net, im, fm,
                        variant=fit_eval.Variants.TOKEN_BASED, parameters=P)["log_fitness"], 4)
                    r["generalization"] = round(gen_eval.apply(log, net, im, fm), 4)

                    if notion == "user" and not COMPUTE_USER_PRECISION:
                        r["precision"] = np.nan
                        r["prec_s"] = np.nan
                        r["prec_note"] = "Skipped by design: ETC precision on long user-level traces is computationally prohibitive."
                    else:
                        try:
                            t0 = time.time()
                            r["precision"] = round(prec_eval.apply(log, net, im, fm,
                                variant=prec_eval.Variants.ETCONFORMANCE_TOKEN, parameters=P), 4)
                            r["prec_s"] = round(time.time() - t0, 1)
                        except Exception as e:
                            r["precision"] = np.nan
                            r["prec_note"] = f"{type(e).__name__}: {str(e)[:100]}"

                    print(f"   {miner:<10} fit={r['fitness']:.4f} prec={r.get('precision')} gen={r['generalization']:.4f} simp={r['simplicity']:.4f} silent={r['silent']}", flush=True)
                except Exception as e:
                    r["error"] = f"{type(e).__name__}: {str(e)[:160]}"
                    print(f"   {miner:<10} FAILED {r['error']}", flush=True)

                results.append(r)
                flush()
                done.add((fac, role, notion, miner))
                results.clear()
                gc.collect()

            del log, ldf
            gc.collect()
        if df is not None:
            del df
        gc.collect()

quality = pd.read_csv(OUT)
print(f"\n{len(quality)} rows saved persistently -> {OUT}")
quality

resuming FIF: 2 combinations already computed from /content/drive/MyDrive/ProcessMining/results/02_quality_FIF.csv

[FIF Student course] 75 traces, 23,982 events, mean len 320


replaying log with TBR, completed traces ::   0%|          | 0/75 [00:00<?, ?it/s]

   inductive  fit=0.9999 prec=0.049 gen=0.7659 simp=0.6285 silent=194

[FIF Student user] 40 traces, 31,529 events, mean len 788


replaying log with TBR, completed traces ::   0%|          | 0/38 [00:00<?, ?it/s]

   alpha      fit=0.0766 prec=nan gen=0.6990 simp=1.0000 silent=0


replaying log with TBR, completed traces ::   0%|          | 0/38 [00:00<?, ?it/s]

   heuristic  fit=0.9741 prec=nan gen=0.6566 simp=0.4624 silent=156


replaying log with TBR, completed traces ::   0%|          | 0/38 [00:00<?, ?it/s]

   inductive  fit=1.0000 prec=nan gen=0.7676 simp=0.6284 silent=185
  [skip] FIF_Lecturer.csv not in deposit

6 rows saved persistently -> /content/drive/MyDrive/ProcessMining/results/02_quality_FIF.csv


,faculty,role,notion,miner,traces,events,activities,mean_len,sample_requested,seed,...,places,transitions,arcs,silent,simplicity,fitness,generalization,precision,prec_s,prec_note
0,FIF,Student,course,alpha,75,23982,48,319.8,75,42,...,12,48,46,0,1.0000,0.1031,0.7402,0.1191,52.1,NaN
1,FIF,Student,course,heuristic,75,23982,48,319.8,75,42,...,86,198,448,150,0.4641,0.9735,0.6439,0.1331,408.6,NaN
2,FIF,Student,course,inductive,75,23982,48,319.8,75,42,...,164,242,526,194,0.6285,0.9999,0.7659,0.0490,5189.5,NaN
3,FIF,Student,user,alpha,40,31529,51,788.2,40,42,...,13,51,45,0,1.0000,0.0766,0.6990,NaN,NaN,Skipped by design: ETC precision on long user-...
4,FIF,Student,user,heuristic,40,31529,51,788.2,40,42,...,82,207,457,156,0.4624,0.9741,0.6566,NaN,NaN,Skipped by design: ETC precision on long user-...
5,FIF,Student,user,inductive,40,31529,51,788.2,40,42,...,153,236,504,185,0.6284,1.0000,0.7676,NaN,NaN,Skipped by design: ETC precision on long user-...


## 4. Does precision change the ranking?

The key question. If Inductive Miner tops fitness but bottoms precision, the
original recommendation does not survive.

In [7]:

import glob
parts = []
for p in sorted(glob.glob(os.path.join(RESULTS_DIR, "02_quality_*.csv"))):
    try:
        parts.append(pd.read_csv(p))
    except Exception as e:
        print("could not read", p, e)

if parts:
    quality_all = pd.concat(parts, ignore_index=True)
    quality_all.drop_duplicates(subset=["faculty","role","notion","miner"], keep="last", inplace=True)
    MASTER = os.path.join(RESULTS_DIR, "02_quality.csv")
    quality_all.to_csv(MASTER, index=False)
    print(f"master table: {len(quality_all)} rows -> {MASTER}")
else:
    quality_all = quality.copy()

q = quality_all[quality_all.error.isna()] if "error" in quality_all else quality_all
for notion in q.notion.unique():
    sub = q[q.notion == notion]
    print(f"\n=== {notion}-level: mean across completed faculty-role partitions ===")
    agg = sub.groupby("miner")[["fitness", "precision", "generalization", "simplicity"]].mean().round(4)
    print(agg.to_string())
    if agg["precision"].notna().all():
        print(f"  best by fitness  : {agg['fitness'].idxmax()}")
        print(f"  best by precision: {agg['precision'].idxmax()}")
        if agg["fitness"].idxmax() != agg["precision"].idxmax():
            print("  -> RANKING REVERSES when precision is included")

master table: 42 rows -> /content/drive/MyDrive/ProcessMining/results/02_quality.csv

=== course-level: mean across completed faculty-role partitions ===
           fitness  precision  generalization  simplicity
miner                                                    
alpha       0.1047     0.1470          0.7300      1.0000
heuristic   0.9770     0.1408          0.6664      0.4700
inductive   0.9955     0.0548          0.7936      0.6138
  best by fitness  : inductive
  best by precision: alpha
  -> RANKING REVERSES when precision is included

=== user-level: mean across completed faculty-role partitions ===
           fitness  precision  generalization  simplicity
miner                                                    
alpha       0.1050        NaN          0.7340      1.0000
heuristic   0.9819        NaN          0.6509      0.4712
inductive   0.9962        NaN          0.7609      0.6103


## 5. LaTeX output

In [8]:
for role in ["Lecturer", "Student"]:
    for notion in q.notion.unique():
        sub = q[(q.role == role) & (q.notion == notion)]
        if not len(sub):
            continue
        print(f"\n% ---- {role}, {notion}-level case notion ----")
        for fac in FACULTIES:
            r = sub[sub.faculty == fac].set_index("miner")
            if not len(r):
                continue
            def g(m, c):
                try:
                    v = r.loc[m, c]
                    return f"{v:.3f}" if pd.notna(v) else "--"
                except Exception:
                    return "--"
            cells = []
            for c_ in ["fitness", "precision", "generalization", "simplicity"]:
                cells += [g(m, c_) for m in ["alpha", "heuristic", "inductive"]]
            print(f"{fac} & " + " & ".join(cells) + r" \\")


% ---- Student, course-level case notion ----
FEB & 0.077 & 0.975 & 0.990 & 0.120 & 0.175 & 0.056 & 0.723 & 0.670 & 0.797 & 1.000 & 0.472 & 0.600 \\
FIF & 0.103 & 0.974 & 1.000 & 0.119 & 0.133 & 0.049 & 0.740 & 0.644 & 0.766 & 1.000 & 0.464 & 0.628 \\
FIK & 0.110 & 0.965 & 1.000 & 0.211 & 0.122 & 0.062 & 0.759 & 0.642 & 0.785 & 1.000 & 0.471 & 0.616 \\
FIT & 0.131 & 0.986 & 1.000 & 0.146 & 0.188 & 0.056 & 0.706 & 0.680 & 0.780 & 1.000 & 0.475 & 0.627 \\
FKB & 0.103 & 0.978 & 0.982 & 0.162 & 0.109 & 0.063 & 0.744 & 0.679 & 0.838 & 1.000 & 0.467 & 0.591 \\
FRI & 0.095 & 0.987 & 0.997 & 0.101 & 0.127 & 0.044 & 0.682 & 0.641 & 0.783 & 1.000 & 0.473 & 0.619 \\
FTE & 0.113 & 0.975 & 1.000 & 0.169 & 0.132 & 0.054 & 0.756 & 0.708 & 0.805 & 1.000 & 0.468 & 0.615 \\

% ---- Student, user-level case notion ----
FEB & 0.100 & 0.977 & 0.988 & -- & -- & -- & 0.748 & 0.636 & 0.754 & 1.000 & 0.477 & 0.596 \\
FIF & 0.077 & 0.974 & 1.000 & -- & -- & -- & 0.699 & 0.657 & 0.768 & 1.000 & 0.462 & 0.628 \\